# Portfolio Project: SpaceX Falcon 9 First-Stage Landing Prediction
## 04 — Exploratory Data Analysis with SQL

**Portfolio Project | Business Analytics & Data Analytics**

This notebook uses **SQL** to investigate the historical SpaceX launch database from several complementary perspectives: launch-site activity, payload mass, customer demand, landing outcomes, booster versions, and time-specific recovery performance.

### Research role of this stage
The objective is not simply to demonstrate SQL syntax. Each query is framed as an **analytical question** that helps characterize the operating environment surrounding **Falcon launches and first-stage recovery**. The results provide context for the visualization, geospatial, dashboard, and predictive-modeling stages that follow.

### Workflow
1. Load the local `Spacex.csv` source file.
2. Create a local SQLite database.
3. Write the CSV into a staging table (`SPACEXTBL`).
4. Create the cleaned analysis table (`SPACEXTABLE`) by removing rows with missing dates.
5. Validate the table schema and record count.
6. Perform SQL-based exploratory analysis.
7. Close the database connection.

> **Data-scope note.** This SQL dataset contains **101 launch records and 10 variables** and is a separate IBM historical snapshot from the 90-launch Falcon 9 modeling dataset created in Notebooks 01–03. It is used here specifically for the SQL exploratory-analysis component of the portfolio.

## 1. Load the local SQL source dataset

The portfolio version uses the locally stored `Spacex.csv` file downloaded from the IBM server. This makes the notebook self-contained.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

CSV_PATH = Path("Spacex.csv")
DB_PATH = Path("spacex_sql_portfolio.db")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"{CSV_PATH} was not found. Place Spacex.csv in the same directory as this notebook."
    )

df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns from {CSV_PATH}")
df.head()

Loaded 101 rows × 10 columns from Spacex.csv


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## 2. Create the SQLite database and staging table

We first import the CSV into a staging table named `SPACEXTBL` with standard Python and SQLite.

In [2]:
connection = sqlite3.connect(DB_PATH)

STAGING_TABLE = "SPACEXTBL"
ANALYSIS_TABLE = "SPACEXTABLE"

df.to_sql(
    STAGING_TABLE,
    connection,
    if_exists="replace",
    index=False,
    method="multi"
)

print(f"Created staging table: {STAGING_TABLE}")

Created staging table: SPACEXTBL


### Create the cleaned analysis table

Remove rows with blank dates before beginning SQL analysis.

In [3]:
cursor = connection.cursor()

cursor.execute(f"DROP TABLE IF EXISTS {ANALYSIS_TABLE};")
cursor.execute(
    f'''
    CREATE TABLE {ANALYSIS_TABLE} AS
    SELECT *
    FROM {STAGING_TABLE}
    WHERE Date IS NOT NULL;
    '''
)

connection.commit()

print(f"Created analysis table: {ANALYSIS_TABLE}")

Created analysis table: SPACEXTABLE


## 3. Database validation

A small helper executes read-only SQL queries and returns the result as a pandas dataframe. Before analysis, the notebook verifies the available tables, schema, row count, and initial records.

In [4]:
def run_query(sql, params=None):
    """Execute a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, connection, params=params)


tables = run_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
)
tables

,name
0,SPACEXTABLE
1,SPACEXTBL


In [5]:
schema = run_query(f"PRAGMA table_info({ANALYSIS_TABLE});")
schema[["name", "type", "notnull", "pk"]]

,name,type,notnull,pk
0,Date,TEXT,0,0
1,Time (UTC),TEXT,0,0
2,Booster_Version,TEXT,0,0
3,Launch_Site,TEXT,0,0
4,Payload,TEXT,0,0
5,PAYLOAD_MASS__KG_,INT,0,0
6,Orbit,TEXT,0,0
7,Customer,TEXT,0,0
8,Mission_Outcome,TEXT,0,0
9,Landing_Outcome,TEXT,0,0


In [6]:
record_count = run_query(
    f"SELECT COUNT(*) AS record_count FROM {ANALYSIS_TABLE};"
)
record_count

,record_count
0,101


In [7]:
run_query(f"SELECT * FROM {ANALYSIS_TABLE} LIMIT 5;")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


The validated `SPACEXTABLE` contains **101 historical launch records** and the 10 fields required for the original SQL analysis.

## 4. Query 1 — Which launch sites are represented?

The first query identifies the unique launch-site labels contained in the database.

In [8]:
query_1 = '''
SELECT DISTINCT Launch_Site
FROM SPACEXTABLE
ORDER BY Launch_Site;
'''

launch_sites = run_query(query_1)
launch_sites

,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


### Interpretation

The database contains four launch-site labels:

- **CCAFS LC-40**
- **CCAFS SLC-40**
- **KSC LC-39A**
- **VAFB SLC-4E**

The two Cape Canaveral labels reflect historical naming conventions within the source dataset.

## 5. Query 2 — What do the first Cape Canaveral records look like?

A prefix filter demonstrates SQL pattern matching and returns the first five records whose launch-site label begins with `CCA`.

In [9]:
query_2 = '''
SELECT *
FROM SPACEXTABLE
WHERE Launch_Site LIKE 'CCA%'
ORDER BY Date
LIMIT 5;
'''

cca_records = run_query(query_2)
cca_records

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


These early records show the initial Falcon 9 operating history at Cape Canaveral, including demonstration and NASA cargo missions.

## 6. Query 3 — How much payload mass was carried for NASA Commercial Resupply Services?

This aggregation measures cumulative payload mass for the `NASA (CRS)` customer category.

In [10]:
query_3 = '''
SELECT
    SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Customer = 'NASA (CRS)';
'''

nasa_crs_payload = run_query(query_3)
nasa_crs_payload

,Total_Payload_Mass_kg
0,45596


### Interpretation

NASA Commercial Resupply Services missions account for a cumulative **45,596 kg** of payload mass in this historical database.

## 7. Query 4 — What was the average payload mass for Falcon 9 v1.1?

The booster field includes both the vehicle version and the individual booster identifier, so `LIKE 'F9 v1.1%'` captures all Falcon 9 v1.1 records.

In [11]:
query_4 = '''
SELECT
    ROUND(AVG(PAYLOAD_MASS__KG_), 2) AS Average_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Booster_Version LIKE 'F9 v1.1%';
'''

f9_v11_avg_payload = run_query(query_4)
f9_v11_avg_payload

,Average_Payload_Mass_kg
0,2534.67


### Interpretation

Falcon 9 v1.1 missions carried an average payload mass of approximately **2,534.67 kg** in this dataset.

## 8. Query 5 — When did the first successful ground-pad landing occur?

The earliest `Success (ground pad)` record identifies the first successful ground return in this SQL snapshot.

In [12]:
query_5 = '''
SELECT
    MIN(Date) AS First_Successful_Ground_Landing
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (ground pad)';
'''

first_ground_landing = run_query(query_5)
first_ground_landing

,First_Successful_Ground_Landing
0,2015-12-22


### Interpretation

The first successful ground-pad landing occurred on **2015-12-22**.

## 9. Query 6 — Which boosters successfully landed on a drone ship with payload mass between 4,000 and 6,000 kg?

The query combines landing outcome and payload constraints. `DISTINCT` is used because the analytical question asks for unique booster records rather than repeated mission rows.

In [13]:
query_6 = '''
SELECT DISTINCT
    Booster_Version
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000
ORDER BY Booster_Version;
'''

drone_ship_boosters = run_query(query_6)
drone_ship_boosters

,Booster_Version
0,F9 FT B1021.2
1,F9 FT B1031.2
2,F9 FT B1022
3,F9 FT B1026


### Interpretation

Four booster records satisfy the specified conditions:

- F9 FT B1021.2
- F9 FT B1022
- F9 FT B1026
- F9 FT B1031.2

This illustrates multi-condition SQL filtering across vehicle, payload, and landing variables.

## 10. Query 7 — How many missions succeeded versus failed?

The raw `Mission_Outcome` field includes several success labels. A `CASE` expression consolidates these into analytically meaningful `Success` and `Failure` groups.

In [14]:
query_7 = '''
SELECT
    CASE
        WHEN Mission_Outcome LIKE 'Success%' THEN 'Success'
        ELSE 'Failure'
    END AS Mission_Status,
    COUNT(*) AS Total_Missions
FROM SPACEXTABLE
GROUP BY Mission_Status
ORDER BY Mission_Status;
'''

mission_summary = run_query(query_7)
mission_summary

,Mission_Status,Total_Missions
0,Failure,1
1,Success,100


### Interpretation

The database contains **100 successful missions and 1 failed mission**. This is a mission-level outcome and should not be confused with the first-stage landing target modeled later in the project.

## 11. Query 8 — Which boosters carried the maximum payload mass?

A scalar subquery first identifies the maximum payload mass, then the outer query returns all booster versions tied at that value.

In [15]:
query_8 = '''
SELECT DISTINCT
    Booster_Version,
    PAYLOAD_MASS__KG_ AS Payload_Mass_kg
FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (
    SELECT MAX(PAYLOAD_MASS__KG_)
    FROM SPACEXTABLE
)
ORDER BY Booster_Version;
'''

max_payload_boosters = run_query(query_8)
max_payload_boosters

,Booster_Version,Payload_Mass_kg
0,F9 B5 B1048.4,15600
1,F9 B5 B1048.5,15600
2,F9 B5 B1049.4,15600
3,F9 B5 B1049.5,15600
4,F9 B5 B1049.7,15600
5,F9 B5 B1051.3,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1051.6,15600
8,F9 B5 B1056.4,15600
9,F9 B5 B1058.3,15600


### Interpretation

The maximum recorded payload mass is **15,600 kg**. Multiple booster records carried missions at this maximum, so returning all matches is analytically preferable to assuming a unique maximum row.

## 12. Query 9 — Which 2015 launches failed to land on a drone ship?

This query identifies failed drone-ship landing attempts in 2015 together with date, booster version, and launch site.

In [16]:
query_9 = '''
SELECT
    Date,
    strftime('%m', Date) AS Month,
    Landing_Outcome,
    Booster_Version,
    Launch_Site
FROM SPACEXTABLE
WHERE strftime('%Y', Date) = '2015'
  AND Landing_Outcome = 'Failure (drone ship)'
ORDER BY Date;
'''

failed_drone_2015 = run_query(query_9)
failed_drone_2015

,Date,Month,Landing_Outcome,Booster_Version,Launch_Site
0,2015-01-10,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,2015-04-14,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Interpretation

Two failed drone-ship landing attempts are recorded in 2015:

- **2015-01-10 — F9 v1.1 B1012**
- **2015-04-14 — F9 v1.1 B1015**

Both missions launched from CCAFS LC-40.

## 13. Query 10 — How were landing outcomes distributed from 2010-06-04 through 2017-03-20?

Group on `Landing_Outcome` and order by frequency.

In [17]:
query_10 = '''
SELECT
    Landing_Outcome,
    COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC, Landing_Outcome;
'''

landing_outcome_ranking = run_query(query_10)
landing_outcome_ranking

,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Failure (drone ship),5
2,Success (drone ship),5
3,Controlled (ocean),3
4,Success (ground pad),3
5,Failure (parachute),2
6,Uncontrolled (ocean),2
7,Precluded (drone ship),1


### Interpretation

Within the specified historical interval, `No attempt` is the most frequent landing outcome. Drone-ship successes and failures, ocean outcomes, parachute failures, and successful ground-pad landings illustrate the experimental evolution of first-stage recovery during the early Falcon 9 program.

## 14. Query 11 — Payload and landing outcome

Adds one compact aggregation to connect payload scale directly with recovery outcomes.

In [18]:
query_11 = '''
SELECT
    Landing_Outcome,
    COUNT(*) AS Mission_Count,
    ROUND(AVG(PAYLOAD_MASS__KG_), 2) AS Avg_Payload_kg
FROM SPACEXTABLE
GROUP BY Landing_Outcome
ORDER BY Mission_Count DESC;
'''

payload_by_landing = run_query(query_11)
payload_by_landing

,Landing_Outcome,Mission_Count,Avg_Payload_kg
0,Success,38,9228.97
1,No attempt,21,4568.43
2,Success (drone ship),14,4745.57
3,Success (ground pad),9,3366.00
4,Failure (drone ship),5,2743.40
5,Controlled (ocean),5,3602.40
6,Failure,3,11233.33
7,Uncontrolled (ocean),2,1358.00
8,Failure (parachute),2,0.00
9,Precluded (drone ship),1,1952.00


### Interpretation

Average payload differs across landing-outcome categories. This is descriptive rather than causal evidence, but it motivates the later multivariate analysis of payload, orbit, site, and recovery success.

## 15. SQL EDA summary

The SQL analysis establishes several useful historical facts:

- four launch-site labels are represented in the database;
- NASA CRS missions carried **45,596 kg** of payload in aggregate;
- Falcon 9 v1.1 missions carried about **2,534.67 kg** on average;
- the first successful ground-pad landing occurred on **2015-12-22**;
- four specified boosters successfully landed on drone ships with payload masses between 4,000 and 6,000 kg;
- the database records **100 successful missions and 1 failed mission**;
- the maximum payload mass is **15,600 kg**;
- two failed drone-ship landings occurred in 2015;
- early recovery outcomes were heterogeneous rather than a simple success/failure process;
- payload characteristics vary across landing-outcome categories.

### Why this notebook matters

This stage demonstrates a complete SQL workflow from **CSV ingestion → SQLite database creation → table construction → query-based analysis**. It also shows that query design should follow the analytical question: categorical outcomes sometimes require semantic consolidation, maximum-value questions can return multiple rows, and grouping variables must match the concept being summarized.

## 16. Close the database connection

In [19]:
connection.close()
print("SQLite connection closed.")

SQLite connection closed.


## Project attribution

This portfolio notebook builds on the SQL exploratory-analysis stage of the **IBM Data Science Professional Certificate SpaceX capstone**. The original historical `Spacex.csv` source and analytical questions are retained, while the implementation has been refactored for portability, database provenance, and research-oriented interpretation.